In [ ]:
import pandas as pd
from splink.duckdb.linker import DuckDBLinker

# Simulate Probabilistic Patient Linkage using Splink
# In a real PySpark environment, we would use SparkLinker, 
# but DuckDBLinker is easier for quick local Colab simulation.

data = [
    {"unique_id": 1, "first_name": "John", "last_name": "Smith", "dob": "1980-01-01", "city": "Boston", "source": "EHR-A"},
    {"unique_id": 2, "first_name": "Jon", "last_name": "Smith", "dob": "1980-01-01", "city": "Boston", "source": "EHR-B"},
    {"unique_id": 3, "first_name": "Mary", "last_name": "Jones", "dob": "1992-05-15", "city": "New York", "source": "EHR-A"},
    {"unique_id": 4, "first_name": "Mary", "last_name": "Jones", "dob": "1992-05-15", "city": "NYC", "source": "Claims"}
]
df = pd.DataFrame(data)

settings = {
    "link_type": "dedupe_only",
    "blocking_rules_to_generate_predictions": [
        "l.first_name = r.first_name",
        "l.last_name = r.last_name"
    ],
    "comparisons": [
        {"output_column_name": "first_name", "comparison_levels": [{"sql_condition": "first_name_l IS NULL OR first_name_r IS NULL", "label_for_charts": "Null", "is_null_level": True}, {"sql_condition": "first_name_l = first_name_r", "label_for_charts": "Exact match"}, {"sql_condition": "levenshtein(first_name_l, first_name_r) <= 2", "label_for_charts": "Fuzzy match"}, {"sql_condition": "ELSE", "label_for_charts": "All other"}]},
        {"output_column_name": "last_name", "comparison_levels": [{"sql_condition": "last_name_l IS NULL OR last_name_r IS NULL", "label_for_charts": "Null", "is_null_level": True}, {"sql_condition": "last_name_l = last_name_r", "label_for_charts": "Exact match"}, {"sql_condition": "ELSE", "label_for_charts": "All other"}]},
        {"output_column_name": "dob", "comparison_levels": [{"sql_condition": "dob_l IS NULL OR dob_r IS NULL", "label_for_charts": "Null", "is_null_level": True}, {"sql_condition": "dob_l = dob_r", "label_for_charts": "Exact match"}, {"sql_condition": "ELSE", "label_for_charts": "All other"}]}
    ],
    "retain_matching_columns": True,
    "retain_intermediate_calculation_columns": True
}

linker = DuckDBLinker(df, settings)

linker.estimate_u_using_random_sampling(max_pairs=1e6)
linker.estimate_parameters_using_expectation_maximisation("l.first_name = r.first_name")
linker.estimate_parameters_using_expectation_maximisation("l.last_name = r.last_name")

df_predict = linker.predict(threshold_match_probability=0.8)
clusters = linker.cluster_pairwise_predictions_at_threshold(df_predict, 0.8)

print("LINKAGE CLUSTERS (OMOP PERSON ID MAPPING):")
print(clusters.as_pandas_dataframe()[["cluster_id", "unique_id", "first_name", "last_name", "source"]])
